In [2]:
import numpy as np
import random
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop
from collections import deque
from tensorflow import gather_nd
from tensorflow.keras.losses import MeanSquaredError

# Создаем экземпляр функции потерь
mean_squared_error = MeanSquaredError()

In [3]:
import numpy as np
import random
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import RMSprop
from collections import deque
from tensorflow import gather_nd
from tensorflow.keras.losses import MeanSquaredError

# Создаем экземпляр функции потерь
mean_squared_error = MeanSquaredError()

class DeepQLearning:

    ###########################################################################
    #   START - функция __init__
    ###########################################################################
    # ВХОДНЫЕ ПАРАМЕТРЫ:
    # env - среда Cart Pole
    # gamma - коэффициент дисконтирования
    # epsilon - параметр для epsilon-жадной стратегии
    # numberEpisodes - общее количество эпизодов обучения

    def __init__(self,env,gamma,epsilon,numberEpisodes):
        self.env=env
        self.gamma=gamma
        self.epsilon=epsilon
        self.numberEpisodes=numberEpisodes

        # размерность состояния
        self.stateDimension=4
        # размерность действия
        self.actionDimension=2
        # максимальный размер буфера воспроизведения
        self.replayBufferSize=300
        # размер батча для обучения, случайно выбираемого из буфера воспроизведения
        self.batchReplayBufferSize=100

        # количество эпизодов между обновлениями параметров целевой сети
        self.updateTargetNetworkPeriod=100

        # счетчик для обновления целевой сети
        self.counterUpdateTargetNetwork=0

        # список для хранения суммы наград за каждый эпизод
        self.sumRewardsEpisode=[]

        # буфер воспроизведения
        self.replayBuffer=deque(maxlen=self.replayBufferSize)

        # основная сеть
        self.mainNetwork=self.createNetwork()

        # целевая сеть
        self.targetNetwork=self.createNetwork()

        # копируем веса из основной сети в целевую
        self.targetNetwork.set_weights(self.mainNetwork.get_weights())

        # этот список используется в функции потерь
        self.actionsAppend=[]

    ###########################################################################
    #   END - функция __init__
    ###########################################################################

    ###########################################################################
    # START - функция для определения функции потерь
    def my_loss_fn(self, y_true, y_pred):
        s1,s2=y_true.shape

        # матрица индексов для выбора элементов
        indices=np.zeros(shape=(self.batchReplayBufferSize,2))
        indices[:,0]=np.arange(self.batchReplayBufferSize)
        indices[:,1]=self.actionsAppend

        # вычисляем потери с помощью MeanSquaredError
        loss = mean_squared_error(gather_nd(y_true, indices=indices.astype(int)),
                                gather_nd(y_pred, indices=indices.astype(int)))
        return loss
    ###########################################################################
    #   END - функция my_loss_fn
    ###########################################################################

    ###########################################################################
    #   START - функция createNetwork()
    def createNetwork(self):
        model=Sequential()
        model.add(Dense(128, input_dim=self.stateDimension, activation='relu'))
        model.add(Dense(56, activation='relu'))
        model.add(Dense(self.actionDimension, activation='linear'))
        # компилируем сеть с пользовательской функцией потерь
        model.compile(optimizer=RMSprop(), loss=self.my_loss_fn, metrics=['accuracy'])
        return model
    ###########################################################################
    #   END - функция createNetwork()
    ###########################################################################

    ###########################################################################
    #   START - функция trainingEpisodes()
    def trainingEpisodes(self):
        for indexEpisode in range(self.numberEpisodes):
            rewardsEpisode=[]
            print("Симуляция эпизода {}".format(indexEpisode))

            (currentState,_)=self.env.reset()
            terminalState=False

            while not terminalState:
                action = self.selectAction(currentState, indexEpisode)
                (nextState, reward, terminalState,_,_) = self.env.step(action)
                rewardsEpisode.append(reward)
                self.replayBuffer.append((currentState, action, reward, nextState, terminalState))
                self.trainNetwork()
                currentState=nextState

            print("Сумма наград {}".format(np.sum(rewardsEpisode)))
            self.sumRewardsEpisode.append(np.sum(rewardsEpisode))
    ###########################################################################
    #   END - функция trainingEpisodes()
    ###########################################################################

    ###########################################################################
    #    START - функция для выбора действия
    def selectAction(self, state, index):
        if index<1:
            return np.random.choice(self.actionDimension)

        randomNumber=np.random.random()

        if index>200:
            self.epsilon=0.999*self.epsilon

        if randomNumber < self.epsilon:
            return np.random.choice(self.actionDimension)
        else:
            Qvalues=self.mainNetwork.predict(state.reshape(1,4), verbose=0)
            return np.random.choice(np.where(Qvalues[0,:]==np.max(Qvalues[0,:]))[0])
    ###########################################################################
    #    END - функция выбора действия
    ###########################################################################

    ###########################################################################
    #    START - функция trainNetwork()
    def trainNetwork(self):
        # Проверка, что в буфере воспроизведения достаточно данных для обучения
        if (len(self.replayBuffer)>self.batchReplayBufferSize):
            # Случайная выборка из буфера воспроизведения
            randomSampleBatch=random.sample(self.replayBuffer, self.batchReplayBufferSize)

            # Инициализация батчей для текущего и следующего состояний
            currentStateBatch=np.zeros(shape=(self.batchReplayBufferSize,4))
            nextStateBatch=np.zeros(shape=(self.batchReplayBufferSize,4))

            # Заполнение батчей данными из выборки
            for index,tupleS in enumerate(randomSampleBatch):
                currentStateBatch[index,:]=tupleS[0]  # Текущее состояние
                nextStateBatch[index,:]=tupleS[3]     # Следующее состояние

            # Получение Q-значений для следующего состояния от целевой сети
            QnextStateTargetNetwork=self.targetNetwork.predict(nextStateBatch, verbose=0)
            # Получение Q-значений для текущего состояния от основной сети
            QcurrentStateMainNetwork=self.mainNetwork.predict(currentStateBatch, verbose=0)

            # Подготовка входных и выходных данных для обучения
            inputNetwork=currentStateBatch
            outputNetwork=np.zeros(shape=(self.batchReplayBufferSize,2))
            self.actionsAppend=[]

            # Расчет целевых значений для обучения
            for index,(currentState,action,reward,nextState,terminated) in enumerate(randomSampleBatch):
                # Если эпизод завершен, целевое значение равно награде
                if terminated:
                    y=reward
                # Иначе применяем уравнение Беллмана с учетом будущих наград
                else:
                    y=reward+self.gamma*np.max(QnextStateTargetNetwork[index])

                # Сохранение выполненного действия
                self.actionsAppend.append(action)
                # Копирование текущих Q-значений
                outputNetwork[index]=QcurrentStateMainNetwork[index]
                # Обновление только Q-значения для выполненного действия
                outputNetwork[index,action]=y

            # Обучение основной сети на подготовленных данных
            self.mainNetwork.fit(inputNetwork, outputNetwork,
                               batch_size=self.batchReplayBufferSize,
                               verbose=0, epochs=100)

            # Увеличение счетчика обновлений целевой сети
            self.counterUpdateTargetNetwork+=1
            # Проверка необходимости обновления целевой сети
            if (self.counterUpdateTargetNetwork>(self.updateTargetNetworkPeriod-1)):
                # Копирование весов из основной сети в целевую
                self.targetNetwork.set_weights(self.mainNetwork.get_weights())
                print("Целевая сеть обновлена! Счетчик: {}".format(self.counterUpdateTargetNetwork))
                # Сброс счетчика обновлений
                self.counterUpdateTargetNetwork=0

In [7]:
pip install gymnasium

In [4]:
import gymnasium as gym

# Создание окружения
env = gym.make('CartPole-v1')

gamma = 1  # Коэффициент дисконтирования будущих наград
epsilon = 0.1  # Параметр ε-жадной стратегии (вероятность случайного действия)

# Количество эпизодов для обучения
# ПРИМЕЧАНИЕ: После определенного количества эпизодов, когда параметры будут выучены,
# эпизоды станут длинными. На этом моменте можно остановить обучение, нажав CTRL+C.
numberEpisodes = 100

# Создание объекта класса DeepQLearning с заданными параметрами
LearningQDeep = DeepQLearning(env, gamma, epsilon, numberEpisodes)

# Запуск процесса обучения
LearningQDeep.trainingEpisodes()

# Получение массива с наградами за каждый эпизод
LearningQDeep.sumRewardsEpisode

# Вывод сводки архитектуры основной нейронной сети
LearningQDeep.mainNetwork.summary()

# Сохранение обученной модели
# Важно, так как обучение занимает много времени, а модель понадобится
# в другом файле для визуализации результатов
LearningQDeep.mainNetwork.save("trained_model_temp.h5")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Симуляция эпизода 0
Сумма наград 24.0
Симуляция эпизода 1
Сумма наград 11.0
Симуляция эпизода 2
Сумма наград 9.0
Симуляция эпизода 3
Сумма наград 40.0
Симуляция эпизода 4
Сумма наград 10.0
Симуляция эпизода 5
Сумма наград 9.0
Симуляция эпизода 6
Сумма наград 11.0
Симуляция эпизода 7
Сумма наград 11.0
Симуляция эпизода 8
Сумма наград 9.0
Симуляция эпизода 9
Сумма наград 14.0
Симуляция эпизода 10
Сумма наград 10.0
Симуляция эпизода 11
Сумма наград 10.0
Симуляция эпизода 12
Сумма наград 9.0
Симуляция эпизода 13
Сумма наград 10.0
Симуляция эпизода 14
Сумма наград 10.0
Симуляция эпизода 15
Целевая сеть обновлена! Счетчик: 100
Сумма наград 8.0
Симуляция эпизода 16
Сумма наград 22.0
Симуляция эпизода 17
Сумма наград 11.0
Симуляция эпизода 18
Сумма наград 9.0
Симуляция эпизода 19
Сумма наград 9.0
Симуляция эпизода 20
Сумма наград 9.0
Симуляция эпизода 21
Сумма наград 9.0
Симуляция эпизода 22
Сумма наград 8.0
Симуляция эпизода 23
Сумма наград 11.0
Симуляция эпизода 24
Целевая сеть обновлена! Сч

KeyboardInterrupt: 

In [9]:
# Выводим сводную информацию о структуре модели -
# количество слоев, параметров и архитектуру нейронной сети
LearningQDeep.mainNetwork.summary()

# Сохраняем обученную модель в файл trained_model_temp.h5
# Это критически важно сделать, так как:
# 1. Обучение модели занимает значительное время
# 2. Сохраненная модель будет использоваться в других файлах
# 3. Позволит визуализировать производительность обученной модели

LearningQDeep.mainNetwork.save("trained_model_temp.h5")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │             640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 56)                  │           7,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 2)                   │             114 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 15,958 (62.34 KB)

 Trainable params: 7,978 (31.16 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,980 (31.18 KB)

In [11]:
# Сохраняем модель в любом случае (даже если обучение прервано)
model_filename = "trained_model_temp.h5"
LearningQDeep.mainNetwork.save(model_filename)
print(f"\nМодель сохранена в файл: {model_filename}")

# Показываем статистику обучения
print("\nСтатистика обучения:")
print(f"Максимальная награда: {max(LearningQDeep.sumRewardsEpisode)}")
print(f"Средняя награда последних 10 эпизодов: {np.mean(LearningQDeep.sumRewardsEpisode[-10:])}")


Модель сохранена в файл: trained_model_temp.h5

Статистика обучения:
Максимальная награда: 158.0
Средняя награда последних 10 эпизодов: 54.6


In [ ]:
!pip install MoviePy

In [5]:
import keras

In [6]:
import gymnasium as gym

In [13]:
import numpy as np

loaded_model = keras.models.load_model("/content/trained_model_temp.h5",custom_objects={'my_loss_fn':DeepQLearning.my_loss_fn})

# Инициализируем счетчик наград
sumObtainedRewards=0

# Создаем окружение
env = gym.make("CartPole-v1",render_mode='rgb_array')
# Сбрасываем окружение в начальное состояние
(currentState,prob)=env.reset()

# Настройки для записи видео:
# video_length - длина видео в шагах симуляции (меньше ожидаемой длины эпизода)
# Для значений меньше 450 работает корректно
video_length=400
# Настраиваем запись видео в папку 'stored_video'
env = gym.wrappers.RecordVideo(env, 'stored_video', video_length=video_length)

# Флаг терминального состояния (изначально False)
terminalState=False

# Основной цикл симуляции
while not terminalState:
    # Получаем Q-значения для текущего состояния (вектор 1x2)
    Qvalues=loaded_model.predict(currentState.reshape(1,4))
    # Выбираем действие с максимальным Q-значением
    action=np.random.choice(np.where(Qvalues[0,:]==np.max(Qvalues[0,:]))[0])

    # Применяем выбранное действие в окружении
    (currentState, currentReward, terminalState,_,_) = env.step(action)
    # Суммируем полученные награды
    sumObtainedRewards+=currentReward

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━

In [14]:
env.reset()
env.close()